# NS07 Tutorial B - Friendship Paradox

**Lecture:** NS07 - Degree Distributions, Scale-Free Networks, and Robustness

**Reading:** Feld (1991), "Why Your Friends Have More Friends Than You Do"

## Learning goals
- Use the lecture notation
  $$
  \mu_f = \frac{1}{N} \sum_i k_i, \qquad
  \mu_{fof} = \frac{\sum_i k_i^2}{\sum_i k_i}.
  $$
- Understand why following an edge oversamples hubs.
- Measure the friendship paradox on small social graphs, random graphs, and a real transportation network.
- Connect the paradox to practical tasks such as monitoring, immunization, and sampling.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()


def mean_degree(G):
    degrees = np.array([degree for _, degree in G.degree()], dtype=float)
    return degrees.mean()


def mean_degree_via_edge_following(G):
    degrees = np.array([degree for _, degree in G.degree()], dtype=float)
    if degrees.sum() == 0:
        return 0.0
    return (degrees ** 2).sum() / degrees.sum()


def local_friendship_table(G):
    degrees = dict(G.degree())
    avg_neighbor_degree = nx.average_neighbor_degree(G)
    rows = []
    for node in G.nodes():
        degree = degrees[node]
        mean_neighbor_degree = avg_neighbor_degree[node] if degree > 0 else np.nan
        rows.append({
            'node': node,
            'degree': degree,
            'avg_neighbor_degree': mean_neighbor_degree,
            'paradox': (mean_neighbor_degree > degree) if degree > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def paradox_summary(G):
    local = local_friendship_table(G)
    active = local[local['degree'] > 0]
    return {
        'mu_f': mean_degree(G),
        'mu_fof': mean_degree_via_edge_following(G),
        'ratio': mean_degree_via_edge_following(G) / mean_degree(G),
        'fraction_local_paradox': active['paradox'].mean(),
    }


def degree_biased_distribution(G):
    support, counts, pmf = compute_pmf([degree for _, degree in G.degree()])
    mean_deg = np.average(support, weights=pmf)
    q = support * pmf / mean_deg
    return support, pmf, q


def sample_degrees_uniform_vs_edges(G, sample_size=30, n_trials=300, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    nodes = np.array(list(G.nodes()))
    edges = np.array(list(G.edges()), dtype=object)
    degrees = dict(G.degree())
    uniform_means = []
    edge_means = []

    for _ in range(n_trials):
        sampled_nodes = rng.choice(nodes, size=sample_size, replace=True)
        uniform_means.append(np.mean([degrees[node] for node in sampled_nodes]))

        sampled_edges = edges[rng.integers(0, len(edges), size=sample_size)]
        sampled_endpoints = [edge[rng.integers(0, 2)] for edge in sampled_edges]
        edge_means.append(np.mean([degrees[node] for node in sampled_endpoints]))

    return np.array(uniform_means), np.array(edge_means)


---
## 1. Intuition on a small social graph

We start from a tiny friendship graph. It is small enough that we can inspect each node directly, which makes the paradox easier to understand before moving to larger networks.


In [ ]:
friends = load_friends()
friends_pos = nx.spring_layout(friends, seed=RANDOM_SEED)
friends_degree = dict(friends.degree())

plot_metric(
    friends,
    friends_degree,
    pos=friends_pos,
    with_labels=True,
    colorbar=False,
    min_node_size_px=28,
    max_node_size_px=48,
    title='Small friendship graph: node size and colour encode degree',
)

local_friends = local_friendship_table(friends)
print(local_friends.round(3).to_string(index=False))


**Interpretation.**
- Claire has the highest degree, so she appears in many other people's neighbourhoods.
- Shelly is isolated, so the local friendship paradox is not defined for her. In the table we exclude isolated nodes when we compute the fraction of nodes that experience the paradox.


---
## 2. Global formulas: $\mu_f$ versus $\mu_{fof}$

The lecture distinguishes two averages:
- $\mu_f$: average degree of a uniformly random node,
- $\mu_{fof}$: average degree of a node reached by following a random edge.

The second average is degree-weighted, so hubs count more.


In [ ]:
summary_small = paradox_summary(friends)
print(f"mu_f   = {summary_small['mu_f']:.3f}")
print(f"mu_fof = {summary_small['mu_fof']:.3f}")
print(f"mu_fof / mu_f = {summary_small['ratio']:.3f}")
print(f"fraction of non-isolated nodes with local paradox = {summary_small['fraction_local_paradox']:.3f}")


---
## 3. Why edge-following oversamples hubs

On a heterogeneous network, the degree distribution seen from nodes,
$$
p(k),
$$
is not the same as the degree distribution seen from random edge endpoints,
$$
q(k) = \frac{k p(k)}{\langle k \rangle}.
$$

We use the OpenFlights USA network again because the hub structure is real and visible.


In [ ]:
openflights = load_openflights_usa()
support, pmf, q = degree_biased_distribution(openflights)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(support, pmf, marker='o', markersize=3, linewidth=1.5, color=CATEGORY_PALETTE['blue'], label='p(k): uniform node sampling')
ax.plot(support, q, marker='o', markersize=3, linewidth=1.5, color=CATEGORY_PALETTE['orange'], label='q(k): edge-following sampling')
style_axis(
    ax,
    title='Uniform sampling versus edge-following on OpenFlights USA',
    xlabel='Degree k',
    ylabel='Probability mass',
    xscale='log',
    yscale='log',
    legend=True,
)
plt.show()

open_summary = paradox_summary(openflights)
print(pd.DataFrame([open_summary]).round(3).to_string(index=False))


**Interpretation.**
- Large hubs are rare under uniform node sampling, but they are much less rare under edge-following.
- That is the mechanism behind the friendship paradox: neighbours are sampled through edges, not uniformly from the node set.


---
## 4. Compare network families

The paradox becomes stronger as degree heterogeneity grows. We compare:
- a **regular** graph, where the paradox should disappear,
- an **ER** graph, where it is usually mild,
- a **BA** graph, where hubs strengthen it,
- the real **OpenFlights USA** network.


In [ ]:
n_ref = openflights.number_of_nodes()
avg_degree_ref = mean_degree(openflights)
regular_k = int(round(avg_degree_ref))
er_p = avg_degree_ref / (n_ref - 1)
ba_m = max(1, round(avg_degree_ref / 2))

regular = nx.random_regular_graph(regular_k, n_ref, seed=RANDOM_SEED)
er = nx.erdos_renyi_graph(n_ref, er_p, seed=RANDOM_SEED)
ba = nx.barabasi_albert_graph(n_ref, ba_m, seed=RANDOM_SEED)

rows = []
for name, graph in [
    ('Regular', regular),
    ('ER', er),
    ('BA', ba),
    ('OpenFlights USA', openflights),
]:
    summary = paradox_summary(graph)
    rows.append({
        'network': name,
        'mu_f': summary['mu_f'],
        'mu_fof': summary['mu_fof'],
        'mu_fof / mu_f': summary['ratio'],
        'local paradox fraction': summary['fraction_local_paradox'],
        'kappa': heterogeneity_kappa(graph),
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.round(3).to_string(index=False))


---
## 5. Local experience of the paradox in the airport network

The global inequality $\mu_{fof} \geq \mu_f$ does not say that every node has more connected neighbours than itself. The **local** version depends on where the node sits in the network.

For each airport we compare its own degree to the average degree of its neighbours.


In [ ]:
local_open = local_friendship_table(openflights)
active_open = local_open[local_open['degree'] > 0].copy()

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.scatter(
    active_open['degree'],
    active_open['avg_neighbor_degree'],
    color=CATEGORY_PALETTE['blue'],
    alpha=0.55,
    s=28,
)

lim_low = min(active_open['degree'].min(), active_open['avg_neighbor_degree'].min())
lim_high = max(active_open['degree'].max(), active_open['avg_neighbor_degree'].max())
ax.plot([lim_low, lim_high], [lim_low, lim_high], linestyle='--', color=CATEGORY_PALETTE['red'], linewidth=2)
style_axis(
    ax,
    title='OpenFlights USA: own degree versus average neighbour degree',
    xlabel='Own degree',
    ylabel='Average neighbour degree',
    xscale='log',
    yscale='log',
)
plt.show()

active_open['neighbour_to_self_ratio'] = active_open['avg_neighbor_degree'] / active_open['degree']
print(
    active_open.sort_values('neighbour_to_self_ratio', ascending=False)
    .head(10)
    .round(3)
    .to_string(index=False)
)


**Interpretation.**
- Small or regional airports often connect to hubs, so their neighbours are much busier than they are.
- High-degree hubs can lie below the diagonal because their neighbours may be less connected on average.


---
## 6. Sampling application: monitoring by friends-of-friends

The paradox is useful because edge-following reaches more central nodes. We simulate two monitoring strategies on OpenFlights USA:
- sample airports uniformly,
- sample airports by first choosing a random edge endpoint.

The second strategy is a structural analogue of "monitor the friends of randomly chosen people".


In [ ]:
uniform_means, edge_means = sample_degrees_uniform_vs_edges(openflights, sample_size=30, n_trials=300)

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.boxplot(
    [uniform_means, edge_means],
    tick_labels=['Uniform node sample', 'Edge-following sample'],
    patch_artist=True,
    boxprops={'facecolor': CATEGORY_PALETTE['blue'], 'alpha': 0.45},
    medianprops={'color': CATEGORY_PALETTE['red'], 'linewidth': 2},
)
style_axis(
    ax,
    title='Average sampled degree across 300 monitoring trials',
    ylabel='Mean degree in sample',
)
plt.show()

print(f"Uniform sampling mean degree   : {uniform_means.mean():.2f}")
print(f"Edge-following mean degree    : {edge_means.mean():.2f}")
print(f"Gain factor                   : {edge_means.mean() / uniform_means.mean():.2f}")


## Takeaway

- The friendship paradox is not psychological first; it is a **sampling effect**.
- Uniform node sampling gives $\mu_f$, while edge-following gives the larger quantity $\mu_{fof}$.
- The gap between them grows with heterogeneity and hubs.
- This is why friend-based monitoring can detect outbreaks, influential accounts, or vulnerable hubs faster than uniform node sampling.
